# A4：Self-Reflection 幻覺校正迴圈 — Generator-Evaluator 實作

> **對應 Blog：** FDE 面試準備指南（二十五）Self-Reflection 與幻覺校正迴圈設計  
> **核心目標：** 親手建 Generator-Evaluator 雙節點，看到第一次幻覺、看到反思修正、看到收斂護欄運作。

## 面試情境

> 「在法務合約問答中，工具返回的 JSON 數據包含矛盾的條款。LLM 第一次生成時沒有注意到，產生了嚴重幻覺。你如何設計一個 Self-Reflection 架構，讓 Agent 在輸出最終答案前能自己檢查並校正？如何防止反思機制陷入死循環？」

## 學習路徑

| Part | 內容 | 關鍵觀察 |
|------|------|----------|
| 1 | 製造幻覺（沒有反思的版本） | 看到錯誤答案 |
| 2 | Generator-Evaluator 架構 | 雙節點設計 |
| 3 | 三層收斂護欄 | 防止無限迴圈 |
| 4 | LangGraph 完整實作 | 完整圖結構 |
| 5 | 成本分析 + 白板語言 | |

In [ ]:
import os
import json
from typing import TypedDict, Annotated, Literal
import operator
from dotenv import load_dotenv

load_dotenv()

from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("環境就緒")

## Part 1：製造幻覺（沒有自我反思的版本）

In [ ]:
# 模擬工具回傳的矛盾合約資料
# 這是 Self-Reflection 要解決的核心問題

CONTRADICTORY_CONTRACT = """
=== 合約摘要（工具回傳的資料）===

【第 3 條 — 違約責任】
甲方若未按時交付，乙方得向甲方請求違約金，
金額為合約總價之 10%，即新台幣 五百萬元整（NTD 5,000,000）。

【第 5 條 — 付款條件】
付款分三期：簽約款 30%、驗收款 50%、尾款 20%。
合約總金額：新台幣 三千萬元整（NTD 30,000,000）。

【第 7 條 — 違約補充條款】（修訂版）
依本合約第 3 條，違約金上限修訂為新台幣 五十萬元整（NTD 500,000）。
（注：本條款與第 3 條存在數字衝突，以本條款為準）

=== 合約資料結束 ===
"""

USER_QUESTION = "請問這份合約的違約金是多少？"

print("模擬情境：")
print(f"用戶問題：{USER_QUESTION}")
print("\n合約中存在矛盾：")
print("  第 3 條：違約金 = NTD 5,000,000")
print("  第 7 條：違約金 = NTD 500,000（修訂，與第 3 條衝突）")

In [ ]:
# ❌ 沒有反思的版本（直接一次生成）

async def naive_answer(question: str, context: str) -> str:
    """直接生成，不做任何自我檢查"""
    messages = [
        SystemMessage(content="你是一個專業的法務助理。根據提供的合約資料回答用戶的問題。"),
        HumanMessage(content=f"合約資料：\n{context}\n\n問題：{question}")
    ]
    response = await llm.ainvoke(messages)
    return response.content

print("=" * 50)
print("❌ 沒有反思的版本")
print("=" * 50)

naive_result = await naive_answer(USER_QUESTION, CONTRADICTORY_CONTRACT)
print(f"\nLLM 直接回答：")
print(naive_result)
print("\n⚠️  問題：LLM 可能只注意到其中一條，忽略矛盾！")

## Part 2：Generator-Evaluator 架構設計

In [ ]:
# ✅ State 設計（Append-only 保留完整決策軌跡）

class ReflectionState(TypedDict):
    user_query: str
    contract_context: str
    
    # Append-only：每次反思都保留歷史（方便 Debug）
    draft_answers: Annotated[list[str], operator.add]
    evaluator_feedbacks: Annotated[list[dict], operator.add]
    
    # 護欄計數器（跨節點可見）
    reflection_count: int
    
    # 最終輸出
    final_answer: str | None
    outcome: str  # "success" | "fallback" | "running"

MAX_REFLECTIONS = 2  # 最多反思 2 次（超過就交人工）

print("State 設計：")
print("  draft_answers: Append-only（每次生成都保留）")
print("  evaluator_feedbacks: Append-only（每次評估都保留）")
print("  reflection_count: 護欄計數器")
print(f"  MAX_REFLECTIONS = {MAX_REFLECTIONS}")

In [ ]:
# ✅ Generator Node（生成節點）

async def generator_node(state: ReflectionState) -> dict:
    """生成初稿答案（如果是重試，帶入 feedback）"""
    
    print(f"\n[Generator] 第 {state['reflection_count'] + 1} 次生成")
    
    # 如果有前一次的 feedback，加入 prompt
    feedback_section = ""
    if state["evaluator_feedbacks"]:
        last_feedback = state["evaluator_feedbacks"][-1]
        feedback_section = f"""
⚠️  上次回答被審查員標記有問題：
  問題類型：{last_feedback.get('error_type', '')}
  具體問題：{last_feedback.get('error_detail', '')}
  嚴重程度：{last_feedback.get('severity', '')}

請在本次回答中特別注意以上問題。
"""
    
    messages = [
        SystemMessage(content=f"""你是一個專業的法務助理。
根據提供的合約資料回答用戶的問題。
如果發現矛盾條款，必須明確指出並說明不確定性。
不要猜測，不要選擇其中一個而忽略另一個。
{feedback_section}"""),
        HumanMessage(content=f"合約資料：\n{state['contract_context']}\n\n問題：{state['user_query']}")
    ]
    
    response = await llm.ainvoke(messages)
    draft = response.content
    
    print(f"  初稿長度: {len(draft)} 字")
    
    return {
        "draft_answers": [draft],  # Append
        "outcome": "running"
    }

In [ ]:
# ✅ Evaluator Node（評估節點）
# 關鍵：Evaluator 的工作是「找問題」，不是「給正確答案」

EVALUATOR_PROMPT = """你是一個嚴格的法務品質審查員。
你的任務是找出答案的問題，而不是給出正確答案。
必須以 JSON 格式輸出評估結果。

評估標準：
  contradiction: 答案是否遺漏了文件中的矛盾或衝突
  hallucination: 答案是否提到了文件中沒有的資訊
  incomplete: 答案是否遺漏了關鍵資訊
  wrong_number: 金額、日期等關鍵數字是否有誤

嚴重程度：
  HIGH: 法律事實錯誤（金額錯、條款錯）
  MEDIUM: 不夠完整（遺漏重要資訊）
  LOW: 格式或表達問題

輸出格式（嚴格 JSON）：
{
  "has_error": true/false,
  "error_type": "contradiction|hallucination|incomplete|wrong_number|none",
  "error_detail": "具體說明問題在哪裡",
  "severity": "HIGH|MEDIUM|LOW|NONE"
}"""

async def evaluator_node(state: ReflectionState) -> dict:
    """評估最新的初稿，輸出結構化的錯誤報告"""
    
    latest_draft = state["draft_answers"][-1]  # 最新一次的生成
    
    print(f"\n[Evaluator] 評估第 {len(state['draft_answers'])} 次初稿")
    
    messages = [
        SystemMessage(content=EVALUATOR_PROMPT),
        HumanMessage(content=f"""原始合約資料：
{state['contract_context']}

用戶問題：
{state['user_query']}

待審查的答案：
{latest_draft}

請評估這個答案是否有問題，以 JSON 格式輸出。""")
    ]
    
    response = await llm.ainvoke(messages)
    
    # 解析 JSON 輸出
    try:
        raw = response.content
        # 處理可能的 markdown code block
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        evaluation = json.loads(raw.strip())
    except json.JSONDecodeError:
        # 解析失敗時的 fallback
        evaluation = {"has_error": True, "error_type": "parse_error", 
                     "error_detail": response.content, "severity": "MEDIUM"}
    
    has_error = evaluation.get("has_error", False)
    severity = evaluation.get("severity", "NONE")
    print(f"  has_error={has_error}, severity={severity}")
    if has_error:
        print(f"  問題：{evaluation.get('error_detail', '')}")
    
    return {
        "evaluator_feedbacks": [evaluation],  # Append
        "reflection_count": state["reflection_count"] + 1
    }

print("Evaluator Node 設計完成")

## Part 3：三層收斂護欄 + Conditional Router

In [ ]:
# ✅ Conditional Router（決定下一步）

def reflection_router(state: ReflectionState) -> Literal["generator", "fallback", "output"]:
    """
    三層護欄：
    1. 次數上限（MAX_REFLECTIONS）
    2. 評估結果（有沒有錯誤）
    3. 嚴重程度（LOW 可以接受輸出）
    """
    latest_feedback = state["evaluator_feedbacks"][-1] if state["evaluator_feedbacks"] else {}
    has_error = latest_feedback.get("has_error", False)
    severity = latest_feedback.get("severity", "NONE")
    count = state["reflection_count"]
    
    print(f"\n[Router] has_error={has_error}, severity={severity}, count={count}/{MAX_REFLECTIONS}")
    
    # 護欄 1：已達反思次數上限
    if count >= MAX_REFLECTIONS:
        print("  → Fallback（達次數上限）")
        return "fallback"
    
    # 沒有錯誤，或只有低嚴重度問題
    if not has_error or severity == "LOW":
        print("  → Output（通過審查）")
        return "output"
    
    # 有中/高嚴重度錯誤，且還有反思次數
    print("  → Generator（帶 feedback 重試）")
    return "generator"


# ✅ Fallback Node
def fallback_node(state: ReflectionState) -> dict:
    """超過反思次數，交人工審核"""
    print("\n[Fallback] 觸發人工審核")
    
    # 在生產環境中，這裡會：
    # 1. 觸發 Cloud Logging Alert
    # 2. 通知 Human Reviewer（Email / Slack）
    # 3. 將 State 快照存入 Firestore
    
    return {
        "final_answer": (
            "⚠️  系統無法得出完全確定的答案。\n"
            "此問題已標記供法務專家人工審核。\n"
            f"反思次數：{state['reflection_count']}，最後評估：{state['evaluator_feedbacks'][-1] if state['evaluator_feedbacks'] else {}}"
        ),
        "outcome": "fallback"
    }


# ✅ Output Node
def output_node(state: ReflectionState) -> dict:
    """審查通過，輸出最終答案"""
    final = state["draft_answers"][-1]  # 最新通過審查的初稿
    return {
        "final_answer": final,
        "outcome": "success"
    }

print("Router + Fallback + Output 設計完成")

## Part 4：LangGraph 完整圖結構

In [ ]:
# ✅ 建立完整的 Self-Reflection Graph

reflection_builder = StateGraph(ReflectionState)

# 加入節點
reflection_builder.add_node("generator", generator_node)
reflection_builder.add_node("evaluator", evaluator_node)
reflection_builder.add_node("fallback", fallback_node)
reflection_builder.add_node("output", output_node)

# 邊的設計
reflection_builder.add_edge(START, "generator")       # 開始 → 生成
reflection_builder.add_edge("generator", "evaluator") # 生成 → 評估（固定）
reflection_builder.add_conditional_edges(             # 評估 → 三條路
    "evaluator",
    reflection_router,
    {
        "generator": "generator",  # 帶 feedback 重試
        "fallback": "fallback",    # 人工兜底
        "output": "output"         # 直接輸出
    }
)
reflection_builder.add_edge("fallback", END)
reflection_builder.add_edge("output", END)

reflection_graph = reflection_builder.compile()

print("✅ Self-Reflection Graph 建立完成")
print("""
圖結構：
  START → generator → evaluator → output → END（通過）
                           ↓
                       generator（帶 feedback 重試，最多 2 次）
                           ↓
                       fallback → END（超過次數）
""")

In [ ]:
# 🎬 執行完整的 Self-Reflection 流程

initial_state: ReflectionState = {
    "user_query": USER_QUESTION,
    "contract_context": CONTRADICTORY_CONTRACT,
    "draft_answers": [],
    "evaluator_feedbacks": [],
    "reflection_count": 0,
    "final_answer": None,
    "outcome": "running"
}

print("=" * 55)
print("🎬 執行 Self-Reflection 迴圈")
print("=" * 55)

final_state = await reflection_graph.ainvoke(initial_state)

print("\n" + "=" * 55)
print("📋 執行摘要")
print("=" * 55)
print(f"結果：{final_state['outcome']}")
print(f"反思次數：{final_state['reflection_count']}")
print(f"生成次數：{len(final_state['draft_answers'])}")
print()
print("最終答案：")
print(final_state["final_answer"])

if len(final_state['draft_answers']) > 1:
    print("\n🔍 第 1 次初稿（反思前）：")
    print(final_state['draft_answers'][0][:500] + "...")

## Part 5：成本分析 + 面試白板語言

In [ ]:
# 成本計算

BASE_COST_PER_QUERY = 0.015  # USD，gpt-4o-mini 每次查詢估算

print("=" * 50)
print("成本分析")
print("=" * 50)
print(f"""
每次反思 = 1 次 Generator + 1 次 Evaluator = ~2 次 LLM 呼叫

不同反思次數的成本（以 gpt-4o-mini 估算）：
  沒有反思（1 次生成）:    ${BASE_COST_PER_QUERY:.3f} / 次查詢
  1 次反思（3 次 LLM）:    ${BASE_COST_PER_QUERY * 3:.3f} / 次查詢（3x）
  2 次反思（5 次 LLM）:    ${BASE_COST_PER_QUERY * 5:.3f} / 次查詢（5x）

如果每天有 1,000 次查詢：
  沒有反思:  ${BASE_COST_PER_QUERY * 1000 * 30:.0f} / 月
  1 次反思:  ${BASE_COST_PER_QUERY * 3 * 1000 * 30:.0f} / 月
  2 次反思:  ${BASE_COST_PER_QUERY * 5 * 1000 * 30:.0f} / 月

結論：反思機制應該是 opt-in，只用在高風險場景
""")

print("什麼場景值得這個成本？")
print("""
  ✅ 值得：
     - 高風險決策（法律、醫療、財務）— 錯誤成本 >> 計算成本
     - 資料有已知衝突可能（多來源整合）
     - 用戶期待高精度（B2B 專業場景）

  ❌ 不值得：
     - 日常 FAQ（已有標準答案）
     - 格式轉換任務（機械性工作）
     - 成本敏感的高頻查詢
""")

In [ ]:
print("""
面試答題框架：
─────────────────────────────────────────────────────
這道題的核心是：設計一個讓 Agent 能自我校正的迴圈，
同時確保這個迴圈一定會終止。

架構設計：Generator-Evaluator 雙節點模式。
  Generator（法務助理角色）產生初稿；
  Evaluator（法務審查員角色，不同 System Prompt）評估初稿，
  輸出結構化的錯誤報告（has_error + error_type + 具體位置 + severity）。
  Conditional Router 根據評估結果決定：
  ├── 通過 → 輸出
  ├── 有問題且可以重試 → 帶 feedback 回到 Generator
  └── 已達上限 → Fallback

收斂保證三層：
  1. MAX_REFLECTIONS = 2（次數上限）
  2. reflection_count 存在 Global State（跨節點可見）
  3. Token Budget Check（防止成本無限燃燒）
  超過上限就跳 Fallback，回傳「需要人工審核」，
  觸發 Cloud Logging Alert。

為什麼 MAX_REFLECTIONS = 2 而不是 5？
  第一次反思解決大部分問題
  第二次反思處理殘留問題
  超過 2 次：問題可能本身無解（矛盾資料），不應繼續
  且每次反思 ≈ 3x 成本，超過 2 次成本不合理

Evaluator 不應該給正確答案：
  ❌「你說 50 萬是錯的，應該是 500 萬」
  ✅「第 3 條寫 500 萬，第 7 條寫 50 萬，答案未提及矛盾」
  Evaluator 找問題，Generator 決定怎麼回應
─────────────────────────────────────────────────────
""")